<a href="https://colab.research.google.com/github/guru16-02/DEEPLEARNING/blob/main/final_isl_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q mediapipe
!pip install -q gTTS
!pip install tensorflow
import os
import sys
import json
import requests
from collections import deque
from pathlib import Path
from typing import List, Optional

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import tensorflow as tf
from gtts import gTTS
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

# --- MediaPipe Tasks API ---
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

print(f"TensorFlow Version: {tf.__version__}")
print(f"MediaPipe Version: {mp.__version__}")
print("All dependencies are installed and libraries are imported.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.23.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
wandb 0.28.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
TensorFlow Version: 2.20.0
MediaPipe Version: 1.0.1
All dependencies are installed and libraries are imported.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- 3. Data Splitting ---

# Define constants for clarity and easy modification
TEST_SET_SIZE = 0.15
VALIDATION_SET_SIZE = 0.15
RANDOM_STATE = 42
MIN_SAMPLES_PER_CLASS = 10 # Each class needs at least 3 samples for a 3-way stratified split

def create_and_save_splits(
    data_path: Path,
    output_dir: Path,
    test_size: float = TEST_SET_SIZE,
    val_size: float = VALIDATION_SET_SIZE,
    file_glob_list: Optional[List[str]] = None,
):
    """
    Collects file paths, creates stratified train/val/test splits, and saves them.
    Filters out classes with too few samples to allow for stratification.
    """
    print(f"\n--- Processing directory: '{data_path}' ---")

    if file_glob_list is None:
        print("Error: file_glob_list must be provided.")
        return

    output_dir.mkdir(parents=True, exist_ok=True)

    if not data_path.exists():
        print(f"Error: Source data path not found at '{data_path}'")
        return

    filepaths = []
    labels = []
    print(f"Searching for files with patterns: {file_glob_list}")
    for class_dir in data_path.iterdir():
        if class_dir.is_dir():
            label = class_dir.name
            for glob_pattern in file_glob_list:
                for filepath in class_dir.glob(glob_pattern):
                    filepaths.append(str(filepath))
                    labels.append(label)

    if not filepaths:
        print(f"Error: No files found in '{data_path}' with the given patterns.")
        return

    paths_df = pd.DataFrame({'filepath': filepaths, 'label': labels})
    print(f"Found {len(paths_df)} total samples across {paths_df['label'].nunique()} classes.")

    # Filter out classes with fewer samples than required for splitting
    class_counts = paths_df['label'].value_counts()
    classes_to_remove = class_counts[class_counts < MIN_SAMPLES_PER_CLASS].index

    if not classes_to_remove.empty:
        print(f"Warning: The following classes have fewer than {MIN_SAMPLES_PER_CLASS} samples and will be excluded: {list(classes_to_remove)}")
        filtered_df = paths_df[~paths_df['label'].isin(classes_to_remove)]
        print(f"         Removed {len(paths_df) - len(filtered_df)} samples.")
    else:
        filtered_df = paths_df

    if len(filtered_df) < MIN_SAMPLES_PER_CLASS:
        print("Error: Not enough data to perform a split after filtering.")
        return

    # Stratified split
    try:
        train_df, temp_df = train_test_split(
            filtered_df,
            test_size=(test_size + val_size),
            stratify=filtered_df['label'],
            random_state=RANDOM_STATE,
        )

        relative_test_size = test_size / (test_size + val_size)
        val_df, test_df = train_test_split(
            temp_df,
            test_size=relative_test_size,
            stratify=temp_df['label'],
            random_state=RANDOM_STATE,
        )
    except ValueError as e:
        print(f"Error during splitting: {e}")
        print("This can happen if a class has too few members for all splits.")
        return

    print(f"Split complete: Train={len(train_df)}, Validation={len(val_df)}, Test={len(test_df)}")

    # Save the splits
    train_df.to_csv(output_dir / 'train_split.csv', index=False)
    val_df.to_csv(output_dir / 'val_split.csv', index=False)
    test_df.to_csv(output_dir / 'test_split.csv', index=False)
    print(f"Splits saved to '{output_dir}'")

# --- Define Paths and Run Splitting ---
static_data_path = Path("/content/drive/MyDrive/archive (2)/Indian")
word_data_path = Path("/content/drive/MyDrive/archive (1)/ProcessedData_vivit")
artifacts_path = Path("./artifacts")

static_file_globs = ['*.[jJ][pP][gG]', '*.[jJ][pP][eE][gG]']
word_file_globs = ['*.[mM][oO][vV]', '*.[mM][pP]4']

create_and_save_splits(static_data_path, artifacts_path / 'static_splits', file_glob_list=static_file_globs)
create_and_save_splits(word_data_path, artifacts_path / 'word_splits', file_glob_list=word_file_globs)

print("\nData splitting process finished.")



--- Processing directory: '/content/drive/MyDrive/archive (2)/Indian' ---
Searching for files with patterns: ['*.[jJ][pP][gG]', '*.[jJ][pP][eE][gG]']
Found 10424 total samples across 9 classes.
Split complete: Train=7296, Validation=1564, Test=1564
Splits saved to 'artifacts/static_splits'

--- Processing directory: '/content/drive/MyDrive/archive (1)/ProcessedData_vivit' ---
Searching for files with patterns: ['*.[mM][oO][vV]', '*.[mM][pP]4']
Found 1166 total samples across 76 classes.
         Removed 108 samples.
Split complete: Train=740, Validation=159, Test=159
Splits saved to 'artifacts/word_splits'

Data splitting process finished.


In [ ]:
# --- 4. Static Model (Alphabets & Numbers) ---

# --- 4.1. Preprocessing ---
IMAGE_SIZE = (224, 224)

def build_augmentation_pipeline(horizontal_flip=False):
    layers = [
        tf.keras.layers.RandomRotation(factor=0.1),
        tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
        tf.keras.layers.RandomZoom(height_factor=0.1, width_factor=0.1),
        tf.keras.layers.RandomContrast(factor=0.2),
        tf.keras.layers.RandomBrightness(factor=0.2),
    ]
    if horizontal_flip:
        layers.append(tf.keras.layers.RandomFlip("horizontal"))
    return tf.keras.Sequential(layers, name="image_augmentation")

def load_and_preprocess_image(path, label, augment=False, augmentation_pipeline=None):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    if augment and augmentation_pipeline is not None:
        image = tf.expand_dims(image, axis=0)
        image = augmentation_pipeline(image, training=True)
        image = tf.squeeze(image, axis=0)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

def create_static_dataset(csv_path, class_map, batch_size, augment=False, shuffle=False):
    df = pd.read_csv(csv_path)
    filepaths = df['filepath'].values
    labels = df['label'].map(class_map).values
    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(df), reshuffle_each_iteration=True)
    augmentation_pipeline = build_augmentation_pipeline(horizontal_flip=False) if augment else None
    dataset = dataset.map(
        lambda path, label: load_and_preprocess_image(path, label, augment, augmentation_pipeline),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

# --- 4.2. Training ---
def train_static_model():
    # Configuration
    ARTIFACTS_DIR = Path("artifacts")
    STATIC_SPLITS_DIR = ARTIFACTS_DIR / "static_splits"
    MODELS_DIR = Path("models")
    MODELS_DIR.mkdir(exist_ok=True)
    BATCH_SIZE = 32
    EPOCHS = 25
    LEARNING_RATE = 0.001

    # Load Data and Class Mappings
    train_df = pd.read_csv(STATIC_SPLITS_DIR / "train_split.csv")
    class_names = sorted(train_df['label'].unique())
    class_map = {name: i for i, name in enumerate(class_names)}
    num_classes = len(class_names)
    print(f"Found {num_classes} static classes.")
    with open(ARTIFACTS_DIR / "static_class_names.json", 'w') as f:
        json.dump(class_names, f)

    # Create Datasets
    train_dataset = create_static_dataset(STATIC_SPLITS_DIR / "train_split.csv", class_map, BATCH_SIZE, augment=True, shuffle=True)
    val_dataset = create_static_dataset(STATIC_SPLITS_DIR / "val_split.csv", class_map, BATCH_SIZE)

    # Build Model
    base_model = tf.keras.applications.MobileNetV2(input_shape=IMAGE_SIZE + (3,), include_top=False, weights='imagenet')
    base_model.trainable = False
    inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)

    # Compile and Train
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.summary()
    print("\nStarting training for the static model...")
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    model_checkpoint = tf.keras.callbacks.ModelCheckpoint(filepath=str(MODELS_DIR / "alphabet_number_model.keras"), save_best_only=True, monitor='val_accuracy')

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=[early_stopping, model_checkpoint]
    )

    print("\nStatic model training complete.")
    print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")
    print(f"Model saved to {MODELS_DIR / 'alphabet_number_model.keras'}")
    return history

# Run the training
static_history = train_static_model()


Found 9 static classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 9)              │        11,529 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,269,513 (8.66 MB)

 Trainable params: 11,529 (45.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)


Starting training for the static model...
Epoch 1/25
228/228 ━━━━━━━━━━━━━━━━━━━━ 1643s 7s/step - accuracy: 0.9427 - loss: 0.2415 - val_accuracy: 1.0000 - val_loss: 0.0151
Epoch 2/25
228/228 ━━━━━━━━━━━━━━━━━━━━ 472s 2s/step - accuracy: 0.9992 - loss: 0.0190 - val_accuracy: 1.0000 - val_loss: 0.0056
Epoch 3/25
228/228 ━━━━━━━━━━━━━━━━━━━━ 451s 2s/step - accuracy: 0.9999 - loss: 0.0091 - val_accuracy: 1.0000 - val_loss: 0.0035
Epoch 4/25
228/228 ━━━━━━━━━━━━━━━━━━━━ 469s 2s/step - accuracy: 1.0000 - loss: 0.0052 - val_accuracy: 1.0000 - val_loss: 0.0024
Epoch 5/25
228/228 ━━━━━━━━━━━━━━━━━━━━ 470s 2s/step - accuracy: 0.9997 - loss: 0.0041 - val_accuracy: 1.0000 - val_loss: 0.0024
Epoch 6/25
 54/228 ━━━━━━━━━━━━━━━━━━━━ 4:57 2s/step - accuracy: 1.0000 - loss: 0.0035

In [ ]:
# --- 5. Dynamic Model (Words) - Landmark Extraction ---

def extract_keypoints(results):
    # Remove [0] so you iterate over the full list of landmarks
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks]).flatten() if results.pose_landmarks else np.zeros(33 * 4)

    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks]).flatten() if results.left_hand_landmarks else np.zeros(21 * 3)

    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks]).flatten() if results.right_hand_landmarks else np.zeros(21 * 3)

    return np.concatenate([pose, lh, rh])

def download_file(url, filename):
    print(f"Downloading {filename}...")
    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print("Download complete.")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading file: {e}")
        sys.exit(1)

def process_videos(video_root_path, output_root_path):
    video_root_path = Path(video_root_path)
    output_root_path = Path(output_root_path)
    if not video_root_path.exists():
        print(f"Error: Video data path not found at '{video_root_path}'")
        return

    action_folders = [f for f in video_root_path.iterdir() if f.is_dir()]

    model_path = 'holistic_landmarker.task'
    model_url = 'https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/1/holistic_landmarker.task'
    if not Path(model_path).exists():
        download_file(model_url, model_path)

    base_options = mp_python.BaseOptions(model_asset_path=model_path)
    options = vision.HolisticLandmarkerOptions(   base_options=base_options,
        running_mode=vision.RunningMode.IMAGE
    )


    with vision.HolisticLandmarker.create_from_options(options) as landmarker:
        for action_folder in tqdm(action_folders, desc="Processing Actions"):
            action_name = action_folder.name
            output_action_folder = output_root_path / action_name
            output_action_folder.mkdir(parents=True, exist_ok=True)
            video_files = list(action_folder.glob('*.[mM][pP]4')) + list(action_folder.glob('*.[mM][oO][vV]'))

            for video_path in tqdm(video_files, desc=f"Videos for {action_name}", leave=False):
                output_path = output_action_folder / f"{video_path.stem}.npy"
                if output_path.exists():
                    continue

                cap = cv2.VideoCapture(str(video_path))
                if not cap.isOpened():
                    print(f"Warning: Could not open video {video_path}")
                    continue

                frame_landmarks = []
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret:
                        break
                    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                    results = landmarker.detect(mp_image)
                    keypoints = extract_keypoints(results)
                    frame_landmarks.append(keypoints)
                cap.release()
                np.save(output_path, np.array(frame_landmarks))

# --- Define Paths and Run Extraction ---
VIDEO_DATA_PATH = Path("/content/drive/MyDrive/archive (1)/ProcessedData_vivit")
SEQUENCE_OUTPUT_PATH = Path("./data/sequences")

print(f"Starting landmark extraction from: {VIDEO_DATA_PATH}")
print(f"Saving sequences to: {SEQUENCE_OUTPUT_PATH}")
process_videos(VIDEO_DATA_PATH, SEQUENCE_OUTPUT_PATH)
print("Landmark extraction complete.")


In [ ]:
# --- 6. Dynamic Model (Words) - Preprocessing and Training ---

# --- 6.1. Preprocessing ---
from pathlib import Path
import pandas as pd
SEQUENCE_LENGTH = 32
NUM_FEATURES = 258

def uniform_temporal_subsample(x, num_samples):
    t = tf.shape(x)[0]
    # Create exactly 'num_samples' evenly spaced indices between 0 and t-1
    indices = tf.linspace(0.0, tf.cast(t - 1, tf.float32), num_samples)

    # Round to nearest integer and cast to int32
    indices = tf.cast(tf.math.round(indices), tf.int32)

    return tf.gather(x, indices)

def load_and_preprocess_sequence(path, label):
    landmarks = np.load(path.numpy().decode('utf-8'))
    if len(landmarks) > SEQUENCE_LENGTH:
        landmarks = uniform_temporal_subsample(tf.constant(landmarks, dtype=tf.float32), SEQUENCE_LENGTH).numpy()
    elif len(landmarks) < SEQUENCE_LENGTH:
        padding = np.zeros((SEQUENCE_LENGTH - len(landmarks), landmarks.shape[1]))
        landmarks = np.concatenate([landmarks, padding], axis=0)
    return landmarks.astype('float32'), label

def create_word_dataset(csv_path, class_map, batch_size, shuffle=False):
    df = pd.read_csv(csv_path)
    video_base_path_str = str(Path(r"/content/drive/MyDrive/archive (1)/ProcessedData_vivit"))
    sequence_base_path_str = "data/sequences"

    filepaths = df['filepath'].str.replace(video_base_path_str, sequence_base_path_str, regex=False) \
                              .str.replace(r'\.(MOV|mp4)$', '.npy', regex=True, case=False) \
                              .values
    labels = df['label'].map(class_map).values.astype('int32')
    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(df), reshuffle_each_iteration=True)

    dataset = dataset.map(lambda path, label: tf.py_function(load_and_preprocess_sequence, [path, label], [tf.float32, tf.int32]))

    # Set the shape of the tensors after the py_function
    dataset = dataset.map(lambda x, y: (tf.ensure_shape(x, [SEQUENCE_LENGTH, NUM_FEATURES]), tf.ensure_shape(y, [])))

    dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

# --- 6.2. Training ---
def train_word_model():
    # Configuration
    ARTIFACTS_DIR = Path("artifacts")
    WORD_SPLITS_DIR = ARTIFACTS_DIR / "word_splits"
    MODELS_DIR = Path("models")
    BATCH_SIZE = 16
    EPOCHS = 150
    LEARNING_RATE = 0.0005

    # Load Data and Class Mappings
    train_df = pd.read_csv(WORD_SPLITS_DIR / "train_split.csv")
    class_names = sorted(train_df['label'].unique())
    class_map = {name: i for i, name in enumerate(class_names)}
    num_classes = len(class_names)
    print(f"Found {num_classes} word classes.")
    with open(ARTIFACTS_DIR / "word_class_names.json", 'w') as f:
        json.dump(class_names, f)

    # Create Datasets
    train_dataset = create_word_dataset(WORD_SPLITS_DIR / "train_split.csv", class_map, BATCH_SIZE, shuffle=True)
    val_dataset = create_word_dataset(WORD_SPLITS_DIR / "val_split.csv", class_map, BATCH_SIZE)

    # Build Model
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(SEQUENCE_LENGTH, NUM_FEATURES)),
        tf.keras.layers.Masking(mask_value=0.0),
        tf.keras.layers.LSTM(64, return_sequences=True),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.LSTM(32),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])

    # Compile and Train
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.summary()
    print("\nStarting training for the word model...")
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    model_checkpoint = tf.keras.callbacks.ModelCheckpoint(filepath=str(MODELS_DIR / "word_model.keras"), save_best_only=True, monitor='val_accuracy')

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=[early_stopping, model_checkpoint]
    )

    print("\nWord model training complete.")
    print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")
    print(f"Model saved to {MODELS_DIR / 'word_model.keras'}")
    return history

# Run the training
word_history = train_word_model()


In [ ]:
# --- 7. Model Evaluation ---

def evaluate_static_model():
    print("\n--- Evaluating Static Model ---")
    ARTIFACTS_DIR = Path("artifacts")
    STATIC_SPLITS_DIR = ARTIFACTS_DIR / "static_splits"
    MODELS_DIR = Path("models")

    # Load model and class names
    model = tf.keras.models.load_model(str(MODELS_DIR / "alphabet_number_model.keras"))
    with open(ARTIFACTS_DIR / "static_class_names.json", 'r') as f:
        class_names = json.load(f)
    class_map = {name: i for i, name in enumerate(class_names)}

    # Create test dataset
    test_csv_path = STATIC_SPLITS_DIR / "test_split.csv"
    if not test_csv_path.exists():
        print(f"Test split not found at {test_csv_path}. Skipping evaluation.")
        return

    test_dataset = create_static_dataset(test_csv_path, class_map, batch_size=32)

    # Predict
    y_true = np.concatenate([y for x, y in test_dataset], axis=0)
    y_pred_probs = model.predict(test_dataset)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Print metrics
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    # Plot confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(15, 15))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Static Model Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

def evaluate_word_model():
    print("\n--- Evaluating Word Model ---")
    ARTIFACTS_DIR = Path("artifacts")
    WORD_SPLITS_DIR = ARTIFACTS_DIR / "word_splits"
    MODELS_DIR = Path("models")

    # Load model and class names
    model = tf.keras.models.load_model(str(MODELS_DIR / "word_model.keras"))
    with open(ARTIFACTS_DIR / "word_class_names.json", 'r') as f:
        class_names = json.load(f)
    class_map = {name: i for i, name in enumerate(class_names)}

    # Create test dataset
    test_csv_path = WORD_SPLITS_DIR / "test_split.csv"
    if not test_csv_path.exists():
        print(f"Test split not found at {test_csv_path}. Skipping evaluation.")
        return

    test_dataset = create_word_dataset(test_csv_path, class_map, batch_size=16)

    # Predict
    y_true = np.concatenate([y for x, y in test_dataset], axis=0)
    y_pred_probs = model.predict(test_dataset)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Print metrics
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    # Plot confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(20, 20))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Word Model Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

# Run evaluations
evaluate_static_model()
evaluate_word_model()
